## Fitting hMFC to Shekhar & Rahnev (2021)

In [1]:
import jax
import jax.numpy as jnp
import jax.random as jr
from jax.nn import sigmoid
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import pandas as pd
import seaborn as sns
from fastprogress import progress_bar
from scipy.stats import spearmanr
import statsmodels.api as sm
import numpy as np
import dill


from jax import vmap, lax

from tensorflow_probability.substrates import jax as tfp
tfd = tfp.distributions

from hmfc.model import HierarchicalBernoulliLDS
from hmfc.gibbs import gibbs_step
from hmfc.utils import convert_mean_to_std_ig_params
from hmfc.constants import A_MAX

### Load in dataset

In [2]:
data = pd.read_csv("/vsc-hard-mounts/leuven-data/343/vsc34314/Shekhar_2021.csv")

num_inputs = 5 # stimulus, prev resp, prev conf, prev resp conf (interaction),  prev stimulus


# Put dataset in correct data structure

num_trials_per_subject = jnp.array(data.groupby('subj').size())
max_num_trials = data.groupby('subj').size().max()  # Find the maximum number of observations
num_trials = max_num_trials

inputs, emissions, masks = [], [], []

for i in np.unique(data.subj):
    df = data[data.subj == i]
        
    evidence = jnp.array(df.evidence)  # scaled between -1 and 1
    prevsignabsevi = jnp.array(df.prevsignabsevi)  # interaction prevsign and prevabsevi
    prevresp = jnp.array(df.prevresp)  # -1 left, 1 right
    prevconf = jnp.array(df.prevconf)  # continuous scale between -1 and 1
    prevrespconf = jnp.array(df.prevrespconf)  # interaction between prevresp and prevconf

    resp = jnp.array(df.resp)

    inputs_subj = jnp.vstack([evidence, prevsignabsevi, prevresp, prevconf, prevrespconf]).T
    
    # MEAN CENTERING THE INPUTS! Required for hMFC
    inputs_subj = inputs_subj - jnp.mean(inputs_subj, axis=0)
    emissions_subj = resp
    
    # Create masking variable
    masks_subj = jnp.ones_like(emissions_subj)

    # Check if the subject has fewer observations than the maximum
    if df.shape[0] < max_num_trials:
        # Calculate the number of observations to fill
        num_to_fill = max_num_trials - df.shape[0]
        # Create a matrix of zeros to fill in the missing observations
        zero_input = jnp.zeros((num_to_fill, num_inputs))
        zero_emissions = jnp.zeros(num_to_fill)

        # Append the zero-filled observations
        inputs_subj = jnp.vstack([inputs_subj, zero_input])
        emissions_subj = jnp.concatenate((emissions_subj, zero_emissions))
        masks_subj = jnp.concatenate((masks_subj, zero_emissions))

    # Append the results to the output lists
    inputs.append(inputs_subj)
    emissions.append(emissions_subj)
    masks.append(masks_subj)

# Convert the lists to arrays
inputs = jnp.array(inputs)
emissions = jnp.array(emissions)
masks = jnp.array(masks)



### Initialize some variables

In [3]:
num_chains = 1
num_iters = 2000
num_trials = max_num_trials # set to number trials of subject with most trials, masking takes care of the other subjects
num_subjects = len(inputs)

### Fit model

In [4]:
def initialize_and_fit_model(key):
    
    """
    Initialize model
    """
    key = jr.PRNGKey(key) if isinstance(key, int) else key
    k1, k2, k3, k4, k5 = jr.split(key, 5)
     
    init_mu_a = tfd.Uniform(0.5, 1.0).sample(seed=k1)
    init_sigma_a = tfd.Uniform(0.0, 0.2).sample(seed=k2) # in _hmfc.py we have an upper limit for sigma_a, so don't exceed, otherwise parameter is not updated!
    init_mu_w = tfd.Uniform(-2.0, 2.0).sample(seed=k3, sample_shape=(num_inputs,))
    init_sigma_w = tfd.Uniform(0.0, 1.0).sample(seed=k4, sample_shape=(num_inputs,))
    init_sigma_mu_x = tfd.Uniform(0.25, 1.0).sample(seed=k5)
    init_mu_sigmasq = 5.0
    init_beta_sigmasq = 0.5

    model = HierarchicalBernoulliLDS(num_inputs,init_mu_a, init_sigma_a, init_mu_w, init_sigma_w, init_mu_sigmasq, init_beta_sigmasq, init_sigma_mu_x)
    params, states, _ = model.sample(key, inputs) # sample initial per-subject parameters and states (criterion trajectory)


    """
    Fit model
    """
    lps = jnp.zeros((num_iters,)) # log probability
    
    posterior_samples_mu_a = jnp.zeros((num_iters,))
    posterior_samples_sigma_a = jnp.zeros((num_iters,))
    posterior_samples_mu_w = jnp.zeros((num_iters, num_inputs))
    posterior_samples_sigma_w = jnp.zeros((num_iters, num_inputs))
    posterior_samples_mu_sigmasq = jnp.zeros((num_iters,))
    posterior_samples_beta_sigmasq = jnp.zeros((num_iters,))
    posterior_samples_sigma_mu_x = jnp.zeros((num_iters,))
    
    posterior_samples_a = jnp.zeros((num_iters, num_subjects))
    posterior_samples_sigmasq = jnp.zeros((num_iters, num_subjects))
    posterior_samples_w = jnp.zeros((num_iters, num_subjects, num_inputs))
    posterior_samples_mu_x = jnp.zeros((num_iters, num_subjects))

    posterior_samples_states = jnp.zeros((num_iters, num_subjects, num_trials))
    
    for itr in progress_bar(range(num_iters)):

        this_key, key = jr.split(key)
        lp, states, params, model = gibbs_step(this_key, emissions, masks, states, inputs, params, model)
        
        lps = lps.at[itr].set(lp)

        posterior_samples_mu_a = posterior_samples_mu_a.at[itr].set(sigmoid(model.logit_mu_a))
        posterior_samples_sigma_a = posterior_samples_sigma_a.at[itr].set(jnp.exp(model.log_sigma_a))
        posterior_samples_mu_w = posterior_samples_mu_w.at[itr].set(model.mu_w)
        posterior_samples_sigma_w = posterior_samples_sigma_w.at[itr].set(jnp.exp(model.log_sigma_w))
        posterior_samples_mu_sigmasq = posterior_samples_mu_sigmasq.at[itr].set(jnp.exp(model.log_mu_sigmasq))
        posterior_samples_beta_sigmasq = posterior_samples_beta_sigmasq.at[itr].set(jnp.exp(model.log_beta_sigmasq))
        posterior_samples_sigma_mu_x = posterior_samples_sigma_mu_x.at[itr].set(jnp.exp(model.log_sigma_mu_x))


        posterior_samples_a = posterior_samples_a.at[itr].set(params['a'])
        posterior_samples_sigmasq = posterior_samples_sigmasq.at[itr].set(params['sigmasq'])
        posterior_samples_w = posterior_samples_w.at[itr].set(params['w'])
        posterior_samples_mu_x = posterior_samples_mu_x.at[itr].set(params['mu_x'])
        
        posterior_samples_states = posterior_samples_states.at[itr].set(states)
    

    return posterior_samples_mu_a, posterior_samples_sigma_a, posterior_samples_mu_w, posterior_samples_sigma_w, posterior_samples_mu_sigmasq, posterior_samples_beta_sigmasq, posterior_samples_sigma_mu_x, posterior_samples_a, posterior_samples_sigmasq, posterior_samples_w, posterior_samples_mu_x, posterior_samples_states, lps


### If one chain

In [5]:
posterior_samples_mu_a, posterior_samples_sigma_a, posterior_samples_mu_w, posterior_samples_sigma_w, posterior_samples_mu_sigmasq, posterior_samples_beta_sigmasq, posterior_samples_sigma_mu_x, posterior_samples_a, posterior_samples_sigmasq, posterior_samples_w, posterior_samples_mu_x, posterior_samples_states, lps = initialize_and_fit_model(0)

### If multiple chains

In [ ]:
keys = jnp.arange(num_chains)
posterior_samples_mu_a, posterior_samples_sigma_a, posterior_samples_mu_w, posterior_samples_sigma_w, posterior_samples_mu_sigmasq, posterior_samples_beta_sigmasq, posterior_samples_sigma_mu_x, posterior_samples_a, posterior_samples_sigmasq, posterior_samples_w, posterior_samples_mu_x, posterior_samples_states, lps = vmap(initialize_and_fit_model)(keys)

###  Save the estimated criterion fluctuations by adding them to original dataframe

In [ ]:
# estimated_criterion_fluctuations = []

# for subject in range(num_subjects):
#   estimated_criterion_fluctuations.append(mean_states[subject,:num_trials_per_subject[subject]])

# estimated_criterion_fluctuations = jnp.concatenate(estimated_criterion_fluctuations)
# data['criterion_fluctuations'] = estimated_criterion_fluctuations

# data.to_csv("/vsc-hard-mounts/leuven-data/343/vsc34314/Shekhar_2021_with_criterion_fluctuations.csv", index=False)

### Save environment

In [6]:
file_name = '/vsc-hard-mounts/leuven-data/343/vsc34314/Shekhar_2021.dil'

list_of_variable_names = ("lps", 
  "posterior_samples_mu_a", "posterior_samples_sigma_a",
  "posterior_samples_mu_sigmasq", "posterior_samples_beta_sigmasq",
  "posterior_samples_mu_w", "posterior_samples_sigma_w",
  "posterior_samples_sigma_mu_x", "posterior_samples_mu_x",
  "posterior_samples_a","posterior_samples_sigmasq",
  "posterior_samples_w", "posterior_samples_states",
  "num_trials", "num_inputs","num_trials_per_subject","num_subjects","num_iters", "inputs", "emissions", "masks")


with open(file_name, 'wb') as file:
    dill.dump(list_of_variable_names, file)  # Store all the names first
    
    for variable_name in list_of_variable_names:
        dill.dump(eval(variable_name), file) # Store the objects themselves